In [80]:
import pandas as pd


In [81]:
df = pd.read_csv("IMDB Dataset.csv", engine="python", on_bad_lines="skip")

In [82]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [83]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [84]:
df.drop_duplicates(inplace = True)

In [85]:
df.shape

(30713, 2)

### Preprocessing

### 1.Converting to lowercase

In [86]:
df["review"] = df["review"].str.lower()

### 2. Removing the URLs

In [87]:
import re

#sample_text = "abc is the word, abc"  #abc=> xyz
#new_teat = re.sub("abc","xyz",sample_text)


In [88]:
def remove_urls(text):
    text = re.sub(r"http\S+", " ", text) # (pattern, replacemnt, string)  # eg - http://www.google.com
    return text 

df["review"] = df["review"].apply(remove_urls)  

### 3. Remove punctuations

In [89]:
def remove_punctuations(text):
    text = re.sub( r"[^A-Za-z0-9\s]", " ", text)  # A- Z a-z 0-9 /s
    return text

df["review"] = df["review"].apply(remove_punctuations)      

### 4. Removing HTML tags

In [90]:
def remove_html(text):
    text = re.sub( r"<.*?>", " ", text)  # 
    return text

df["review"] = df["review"].apply(remove_html) 

### 5. Removing  the stopwords

In [91]:
import nltk

nltk.download("punkt")  #  formal tokenizer of nltk
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to C:\Users\Bidisha
[nltk_data]     Das\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Bidisha
[nltk_data]     Das\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Bidisha
[nltk_data]     Das\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [92]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [93]:
#sample_text = "I love coding in python"
#text = word_tokenize(sample_text)

In [94]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")


    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
    return text        

df["review"] = df["review"].apply(remove_stopwords) 

In [95]:
df.head()

,review,sentiment
0,e revewers ned wchg 1 oz epode hooke...,positive
1,wderful ltle producti br br filming t...,positive
2,thought th werful wy pen tme o hot ummer...,positive
3,bclly fmly lttle boy jke thk zombe ...,negative
4,petter mttei love time mey i viully tun...,positive


### 6.stemmimg

In [96]:
# running => run
# played => play
# Porterstemmimg

from nltk.stem import PorterStemmer

In [97]:
def stemmimg(text):
    ps = PorterStemmer()
    stemmed_words = []
    tokens = word_tokenize(text)

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemmimg)    

In [101]:
df.head()

,review,sentiment
0,e revew ned wchg 1 oz epod hook y rgh excli hp...,1
1,wder ltle producti br br film techniqu unum ol...,1
2,thought th wer wy pen tme o hot ummer weeken t...,1
3,bclli fmli lttle boy jke thk zomb h cloet h pn...,0
4,petter mttei love time mey i viulli tunng film...,1


### 7. Encoding


In [98]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [99]:
y = df["sentiment"]

In [100]:
y

0        1
1        1
2        1
3        0
4        1
        ..
30871    0
30872    0
30873    0
30874    1
30875    0
Name: sentiment, Length: 30713, dtype: int64

### 8. vectorization

In [103]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)
X  = tf.fit_transform(df["review"])

### Dataset snd Dataloader

In [106]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [109]:
import torch 
from torch.utils.data import TensorDataset, DataLoader

In [115]:
X_train = X_train.toarray()
X_test = X_test.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [118]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float(),

)


test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float(),

)


In [119]:
train_loader = DataLoader(train_set, shuffle = True, batch_size = 64)
test_loader = DataLoader(test_set, batch_size = 64)

### Build our RNN

In [121]:
import torch.nn as nn
import torch.optim as optim

In [122]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [123]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### Training the RNN

In [124]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.3418675363063812
epoch = 2/10 and loss = 0.2909905016422272
epoch = 3/10 and loss = 0.3330688774585724
epoch = 4/10 and loss = 0.25632575154304504
epoch = 5/10 and loss = 0.10203058272600174
epoch = 6/10 and loss = 0.3675249218940735
epoch = 7/10 and loss = 0.15760251879692078
epoch = 8/10 and loss = 0.25467899441719055
epoch = 9/10 and loss = 0.24279095232486725
epoch = 10/10 and loss = 0.2217937558889389


In [125]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 82.46784958489337
